# Phase 4 — Automated Failure Diagnosis and Recovery

## Objective

The objective of Phase 4 is to develop an automated mechanism that diagnoses detected data-quality failures and applies an appropriate corrective action.

This phase extends the missingness detector developed in Phase 3 by connecting detection results to a diagnosis and recovery engine.

## Research Question

Can an SHML system automatically diagnose feature-specific missingness and apply a reproducible correction without manual intervention?

## Experimental Setup

- **Dataset:** UCI Default of Credit Card Clients
- **Prediction task:** Binary classification
- **Baseline model:** Logistic Regression
- **Monitoring feature:** BILL_AMT1
- **Healthy reference missingness:** 0%
- **Detection threshold:** Greater than 5 percentage points
- **Failure type:** Missing numerical feature values
- **Recovery method:** Median imputation
- **Imputation reference:** Training data only
- **Experimental scenarios:** 0%, 5%, 15%, and 30% missingness

## Diagnosis Strategy

When the monitoring system detects abnormal missingness, the diagnosis engine will:

1. Identify the affected feature.
2. Determine the failure type.
3. Measure the observed missingness.
4. Determine whether the monitoring threshold was exceeded.
5. Select a compatible recovery strategy.

For the initial experiment, the supported diagnosis is numerical-feature missingness in BILL_AMT1.

## Recovery Strategy

The recovery engine will replace missing BILL_AMT1 values using the median calculated from the healthy training dataset.

The median must not be calculated from the corrupted validation batch because that would introduce data leakage and make recovery behavior dependent on the corrupted data.

## Scope and Safety

The original validation data will remain unchanged.

The system will retain an unmodified copy of each corrupted batch and record the recovery action applied.

Recovery will not automatically imply that the corrected batch is safe for use.

Phase 5 will evaluate post-recovery predictive performance and determine whether each recovery should be accepted or rejected.

## Step 4.2 — Baseline Model and Failure Detector Integration

### Objective

Reproduce the healthy baseline model and integrate the feature-specific missingness detector developed in Phase 3.

This establishes the reference components required for automated failure diagnosis and recovery.

### Components

1. **Healthy baseline model:** Logistic regression trained on the original training dataset.
2. **Preprocessing:** StandardScaler fitted exclusively on training data.
3. **Failure detector:** Monitors BILL_AMT1 missingness against the healthy training reference.
4. **Experimental batches:** Validation batches with 0%, 5%, 15%, and 30% missingness.

### Expected Outcome

The reproduced model should match the Phase 3 baseline metrics. The detector should identify the 15% and 30% missingness scenarios while leaving the 0% and 5% scenarios unflagged.

The test dataset will remain untouched.

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [22]:
from pathlib import Path

data_dir = Path("data/working")
candidates = sorted(data_dir.glob("*credit*card*clients*.xls*"))
if not candidates:
    raise FileNotFoundError(f"No dataset file found under {data_dir}")

DATA_PATH = str(candidates[0])
df = pd.read_excel(DATA_PATH, header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [23]:
X = df.drop(columns=["ID", "default payment next month"])
y = df["default payment next month"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (21000, 23)
Validation: (4500, 23)
Test: (4500, 23)


In [24]:

# Reproduce Healthy Baseline
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_scaled, y_train)

y_val_pred = baseline_model.predict(X_val_scaled)
y_val_prob = baseline_model.predict_proba(X_val_scaled)[:, 1]

baseline_metrics = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1-score": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")


Accuracy: 0.8087
Precision: 0.6948
Recall: 0.2402
F1-score: 0.3570
ROC-AUC: 0.7130


In [25]:

def detect_feature_missingness(
    X_batch,
    feature,
    reference_missingness=0.0,
    threshold=0.05
):
    if feature not in X_batch.columns:
        raise ValueError(f"Feature '{feature}' is missing")

    current_missingness = X_batch[feature].isna().mean()

    missingness_increase = (
        current_missingness - reference_missingness
    )

    failure_detected = missingness_increase > threshold

    return {
        "feature": feature,
        "current_missingness": current_missingness,
        "reference_missingness": reference_missingness,
        "missingness_increase": missingness_increase,
        "threshold": threshold,
        "failure_detected": bool(failure_detected)
    }


In [6]:

def inject_feature_missingness(
    X_batch,
    feature,
    missing_rate,
    random_state=42
):
    if feature not in X_batch.columns:
        raise ValueError(f"Unknown feature: {feature}")

    if not 0 <= missing_rate <= 1:
        raise ValueError("missing_rate must be between 0 and 1")

    rng = np.random.default_rng(random_state)
    X_corrupted = X_batch.copy(deep=True)

    n_missing = int(round(len(X_corrupted) * missing_rate))

    selected_positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    feature_col = X_corrupted.columns.get_loc(feature)
    X_corrupted.iloc[selected_positions, feature_col] = np.nan

    return X_corrupted


missingness_levels = [0.00, 0.05, 0.15, 0.30]

corrupted_batches = {}

for rate in missingness_levels:
    corrupted_batches[rate] = inject_feature_missingness(
        X_val,
        feature="BILL_AMT1",
        missing_rate=rate,
        random_state=42
    )

print("Experimental batches recreated successfully.")


Experimental batches recreated successfully.


In [26]:

for rate, X_batch in corrupted_batches.items():

    result = detect_feature_missingness(
        X_batch,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    print(
        f"Missingness: {rate:.0%} | "
        f"Failure detected: {result['failure_detected']}"
    )


Missingness: 0% | Failure detected: False
Missingness: 5% | Failure detected: False
Missingness: 15% | Failure detected: True
Missingness: 30% | Failure detected: True


### Build the Automated Failure Diagnosis Engine

## Step 4.3 — Automated Failure Diagnosis Engine

### Objective

Develop a rule-based diagnosis engine that interprets missingness detection results and recommends an appropriate recovery strategy.

### Diagnosis Inputs

The diagnosis engine receives the output of the Phase 3 missingness detector, including:

- Monitored feature
- Current missingness rate
- Healthy reference missingness rate
- Missingness increase
- Detection threshold
- Failure detection status

### Diagnosis Outputs

The engine produces:

- Failure type
- Affected feature
- Failure severity
- Alert status
- Recommended recovery action
- Diagnosis explanation

### Diagnosis Rules

For BILL_AMT1:

- No missingness: No correction required.
- Missingness at or below 5%: No monitoring alert; routine preprocessing is still required before model inference.
- Missingness greater than 5% and up to 20%: Moderate missingness failure.
- Missingness greater than 20%: Severe missingness failure.

The severity boundaries are preliminary engineering definitions used to demonstrate automated diagnosis.

### Recovery Selection

Because BILL_AMT1 is numerical, the initial recovery strategy is median imputation using the healthy training-data median.

This phase will select a recovery action based on the diagnosis. The actual correction will be implemented in Step 4.4.

### Limitation

The current diagnosis engine supports one failure type: missing numerical values in BILL_AMT1. Other failure types, including numerical drift, categorical shift, and schema violations, will require additional diagnostic rules.

In [27]:

def diagnose_failure(detection_result):
    """
    Diagnose BILL_AMT1 missingness and recommend
    an appropriate recovery action.
    """

    feature = detection_result["feature"]
    missingness = detection_result["current_missingness"]
    failure_detected = detection_result["failure_detected"]

    # Validate supported feature
    if feature != "BILL_AMT1":
        raise ValueError(
            f"No diagnosis rule configured for {feature}"
        )

    # Determine severity and recovery strategy
    if missingness == 0:
        severity = "None"
        failure_type = "No missingness"
        recovery_action = "No correction"

    elif not failure_detected:
        severity = "Below alert threshold"
        failure_type = "Numerical missingness"
        recovery_action = "Routine median preprocessing"

    elif missingness <= 0.20:
        severity = "Moderate"
        failure_type = "Numerical missingness"
        recovery_action = "Median imputation"

    else:
        severity = "Severe"
        failure_type = "Numerical missingness"
        recovery_action = "Median imputation"

    explanation = (
        f"{feature} has {missingness:.2%} missing values. "
        f"Severity: {severity}. "
        f"Recommended action: {recovery_action}."
    )

    return {
        "feature": feature,
        "failure_type": failure_type,
        "severity": severity,
        "failure_detected": failure_detected,
        "recovery_action": recovery_action,
        "explanation": explanation
    }


In [28]:

diagnosis_results = []

for rate, X_batch in corrupted_batches.items():

    detection = detect_feature_missingness(
        X_batch,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    diagnosis = diagnose_failure(detection)

    diagnosis_results.append({
        "Injected Missingness": rate,
        "Failure Detected": diagnosis["failure_detected"],
        "Failure Type": diagnosis["failure_type"],
        "Severity": diagnosis["severity"],
        "Recovery Action": diagnosis["recovery_action"]
    })

diagnosis_df = pd.DataFrame(diagnosis_results)

display(diagnosis_df)


,Injected Missingness,Failure Detected,Failure Type,Severity,Recovery Action
0,0.00,False,No missingness,None,No correction
1,0.05,False,Numerical missingness,Below alert threshold,Routine median preprocessing
2,0.15,True,Numerical missingness,Moderate,Median imputation
3,0.30,True,Numerical missingness,Severe,Median imputation


In [29]:

expected_severities = [
    "None",
    "Below alert threshold",
    "Moderate",
    "Severe"
]

expected_actions = [
    "No correction",
    "Routine median preprocessing",
    "Median imputation",
    "Median imputation"
]

assert diagnosis_df["Severity"].tolist() == expected_severities

assert diagnosis_df["Recovery Action"].tolist() == expected_actions

print("All diagnosis rule checks passed.")


All diagnosis rule checks passed.


## Step 4.4 — Automated Recovery Engine

### Objective

Implement an automated recovery engine that corrects missing values in BILL_AMT1 using a median derived exclusively from healthy training data.

### Recovery Strategy

The recovery engine receives the corrupted validation batch and the diagnosis result.

It selects the correction based on the recommended recovery action:

- No correction: Preserve the original batch.
- Routine median preprocessing: Replace missing values without raising a monitoring alert.
- Median imputation: Apply the same training-derived median to a detected missingness failure.

### Data Leakage Prevention

The replacement value is calculated from the training dataset and is never recalculated using validation or test data.

### Recovery Logging

For each batch, record the injected missingness level, diagnosis severity, recovery action, number of missing values before and after correction, and number of values imputed.

### Scope

This step implements data correction only. Predictive-performance validation and acceptance or rejection decisions will be performed in Phase 5.

In [30]:

# Calculate the reference value using training data only
bill_amt1_median = X_train["BILL_AMT1"].median()

print(f"Training median for BILL_AMT1: {bill_amt1_median:.2f}")


Training median for BILL_AMT1: 22225.50


In [31]:

def recover_missingness(
    X_batch,
    diagnosis,
    training_median
):
    """
    Apply the diagnosed recovery action without
    modifying the original validation batch.
    """

    feature = diagnosis["feature"]
    action = diagnosis["recovery_action"]

    if feature != "BILL_AMT1":
        raise ValueError("Unsupported recovery feature")

    # Preserve the original corrupted batch
    X_recovered = X_batch.copy(deep=True)

    missing_before = int(
        X_recovered[feature].isna().sum()
    )

    supported_actions = {
        "No correction",
        "Routine median preprocessing",
        "Median imputation"
    }

    if action not in supported_actions:
        raise ValueError(f"Unsupported action: {action}")

    if action in {
        "Routine median preprocessing",
        "Median imputation"
    }:
        X_recovered[feature] = (
            X_recovered[feature].fillna(training_median)
        )

    missing_after = int(
        X_recovered[feature].isna().sum()
    )

    recovery_log = {
        "feature": feature,
        "severity": diagnosis["severity"],
        "action": action,
        "missing_before": missing_before,
        "missing_after": missing_after,
        "values_imputed": missing_before - missing_after,
        "recovery_applied": missing_before > missing_after
    }

    return X_recovered, recovery_log


In [32]:

recovered_batches = {}
recovery_logs = []

for rate, X_batch in corrupted_batches.items():

    detection = detect_feature_missingness(
        X_batch,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    diagnosis = diagnose_failure(detection)

    X_recovered, log = recover_missingness(
        X_batch=X_batch,
        diagnosis=diagnosis,
        training_median=bill_amt1_median
    )

    recovered_batches[rate] = X_recovered

    recovery_logs.append({
        "Injected Missingness": rate,
        "Severity": log["severity"],
        "Action": log["action"],
        "Missing Before": log["missing_before"],
        "Missing After": log["missing_after"],
        "Values Imputed": log["values_imputed"]
    })

recovery_df = pd.DataFrame(recovery_logs)

display(recovery_df)


,Injected Missingness,Severity,Action,Missing Before,Missing After,Values Imputed
0,0.00,None,No correction,0,0,0
1,0.05,Below alert threshold,Routine median preprocessing,225,0,225
2,0.15,Moderate,Median imputation,675,0,675
3,0.30,Severe,Median imputation,1350,0,1350


In [33]:

for rate in missingness_levels:

    original_batch = corrupted_batches[rate]
    recovered_batch = recovered_batches[rate]

    # No missing BILL_AMT1 values should remain
    assert recovered_batch["BILL_AMT1"].isna().sum() == 0

    # Original corrupted batches must remain unchanged
    assert (
        original_batch["BILL_AMT1"].isna().sum()
        == round(len(original_batch) * rate)
    )

    # Only originally missing BILL_AMT1 values may change
    originally_present = original_batch["BILL_AMT1"].notna()

    assert (
        recovered_batch.loc[originally_present, "BILL_AMT1"]
        .equals(
            original_batch.loc[originally_present, "BILL_AMT1"]
        )
    )

    # All other features must remain identical
    other_features = original_batch.columns.drop("BILL_AMT1")

    assert recovered_batch[other_features].equals(
        original_batch[other_features]
    )

    print(f"{rate:.0%} scenario: Recovery checks passed")


0% scenario: Recovery checks passed
5% scenario: Recovery checks passed
15% scenario: Recovery checks passed
30% scenario: Recovery checks passed


## Phase 4 — Integration Experiment Log

### Experiment

Integration of feature-specific missingness detection, rule-based diagnosis, and training-median recovery.

### Expected Behavior

The integrated pipeline should reproduce the same detection and recovery results as the individual components.

### Observed Behavior

*Complete after executing the pipeline.*

### What Worked

*Record successful detection, diagnosis, recovery, and consistency checks.*

### What Did Not Work or Required Adjustment

*Record any errors, integration problems, unsupported cases, or design limitations. If no execution errors occurred, state that explicitly rather than inventing failures.*

### Research Limitations

The current implementation supports BILL_AMT1 missingness only.

The severity thresholds are predefined engineering rules.

The recovery mechanism uses median imputation rather than dynamically comparing multiple candidate strategies.

The pipeline has not yet established whether corrected data restores acceptable predictive performance.

### Next Step

Evaluate corrected batches against the healthy baseline and apply explicit recovery acceptance criteria in Phase 5.

## Step 4.6 — Final Pipeline Verification

### Objective

Verify the correctness, consistency, reproducibility, and data integrity of the integrated SHML detection, diagnosis, and recovery pipeline.

### Verification Criteria

The pipeline must:

1. Produce the expected detection decisions for all four missingness scenarios.
2. Select the appropriate diagnosis and recovery action.
3. Correct all injected BILL_AMT1 missing values.
4. Preserve all originally observed values and unrelated features.
5. Leave the original corrupted batches unchanged.
6. Produce identical results when rerun with the same inputs.

Successful verification demonstrates that the implemented pipeline behaves consistently with its predefined rules. It does not yet establish predictive-performance recovery.

In [34]:
def run_shml_recovery_pipeline(
    X_batch,
    training_median,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05,
):
    X_recovered = X_batch.copy(deep=True)

    current_missingness = X_batch[feature].isna().mean()
    failure_detected = bool(
        current_missingness - reference_missingness > threshold
    )

    if failure_detected:
        if pd.isna(training_median):
            raise ValueError("The training median cannot be missing.")

        X_recovered[feature] = (
            X_recovered[feature].fillna(training_median)
        )

    log = {
        "feature": feature,
        "current_missingness": current_missingness,
        "failure_detected": failure_detected,
        "recovery_applied": failure_detected,
        "recovery_method": "median_imputation" if failure_detected else None,
        "remaining_missingness": X_recovered[feature].isna().mean(),
    }

    return X_recovered, log

In [35]:

print("Pipeline function available:",
      callable(run_shml_recovery_pipeline))


Pipeline function available: True


In [37]:

def run_shml_recovery_pipeline(
    X_batch,
    training_median,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
):
    """
    Integrated SHML pipeline:
    Detection -> Diagnosis -> Recovery -> Logging
    """

    # Step 1: Detect
    detection = detect_feature_missingness(
        X_batch=X_batch,
        feature=feature,
        reference_missingness=reference_missingness,
        threshold=threshold
    )

    # Step 2: Diagnose
    diagnosis = diagnose_failure(detection)

    # Step 3: Recover
    X_recovered, recovery_log = recover_missingness(
        X_batch=X_batch,
        diagnosis=diagnosis,
        training_median=training_median
    )

    # Step 4: Build a consistent audit log
    missing_before = int(X_batch[feature].isna().sum())
    missing_after = int(X_recovered[feature].isna().sum())
    values_imputed = missing_before - missing_after

    pipeline_log = {
        "feature": feature,
        "observed_missingness": detection["current_missingness"],
        "missingness_increase": detection["missingness_increase"],
        "failure_detected": detection["failure_detected"],
        "failure_type": diagnosis["failure_type"],
        "severity": diagnosis["severity"],
        "recovery_action": diagnosis["recovery_action"],
        "missing_before": missing_before,
        "missing_after": missing_after,
        "values_imputed": values_imputed,
        "recovery_applied": values_imputed > 0
    }

    return X_recovered, pipeline_log


In [38]:

_, test_log = run_shml_recovery_pipeline(
    X_batch=corrupted_batches[0.15],
    training_median=bill_amt1_median
)

print("Values imputed:", test_log["values_imputed"])
print("Missing after:", test_log["missing_after"])

assert test_log["values_imputed"] == 675
assert test_log["missing_after"] == 0

print("Pipeline function verified successfully.")


Values imputed: 675
Missing after: 0
Pipeline function verified successfully.


In [42]:

pipeline_logs = []

for rate in missingness_levels:

    _, log = run_shml_recovery_pipeline(
        X_batch=corrupted_batches[rate],
        training_median=bill_amt1_median
    )

    pipeline_logs.append({
        "Injected Missingness": rate,
        **log
    })

pipeline_df = pd.DataFrame(pipeline_logs)

display(pipeline_df)


,Injected Missingness,feature,observed_missingness,missingness_increase,failure_detected,failure_type,severity,recovery_action,missing_before,missing_after,values_imputed,recovery_applied
0,0.00,BILL_AMT1,0.00,0.00,False,No missingness,None,No correction,0,0,0,False
1,0.05,BILL_AMT1,0.05,0.05,False,Numerical missingness,Below alert threshold,Routine median preprocessing,225,0,225,True
2,0.15,BILL_AMT1,0.15,0.15,True,Numerical missingness,Moderate,Median imputation,675,0,675,True
3,0.30,BILL_AMT1,0.30,0.30,True,Numerical missingness,Severe,Median imputation,1350,0,1350,True


In [44]:

from pathlib import Path
import pandas as pd

expected_alerts = {
    0.00: False,
    0.05: False,
    0.15: True,
    0.30: True
}

expected_counts = {
    0.00: 0,
    0.05: 225,
    0.15: 675,
    0.30: 1350
}

verification_results = []
pipeline_logs = []

for rate in [0.00, 0.05, 0.15, 0.30]:

    X_original = corrupted_batches[rate]

    X_recovered, log = run_shml_recovery_pipeline(
        X_batch=X_original,
        training_median=bill_amt1_median,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    # Check detection and recovery
    assert log["failure_detected"] == expected_alerts[rate]
    assert log["values_imputed"] == expected_counts[rate]
    assert log["missing_after"] == 0

    # Check original batch was not modified
    assert (
        X_original["BILL_AMT1"].isna().sum()
        == expected_counts[rate]
    )

    # Check all other features are unchanged
    other_features = X_original.columns.drop("BILL_AMT1")
    pd.testing.assert_frame_equal(
        X_recovered[other_features],
        X_original[other_features]
    )

    # Check existing BILL_AMT1 values are unchanged
    observed = X_original["BILL_AMT1"].notna()
    pd.testing.assert_series_equal(
        X_recovered.loc[observed, "BILL_AMT1"],
        X_original.loc[observed, "BILL_AMT1"]
    )

    # Check recovered data is reproducible
    X_repeat, repeat_log = run_shml_recovery_pipeline(
        X_batch=X_original,
        training_median=bill_amt1_median
    )
    pd.testing.assert_frame_equal(X_recovered, X_repeat)
    assert log == repeat_log

    verification_results.append({
        "Missingness": rate,
        "Alert": log["failure_detected"],
        "Severity": log["severity"],
        "Values Imputed": log["values_imputed"],
        "Remaining Missing": log["missing_after"],
        "Verification": "PASS"
    })

    pipeline_logs.append({
        "Injected Missingness": rate,
        **log
    })

# Create both DataFrames
verification_df = pd.DataFrame(verification_results)
pipeline_df = pd.DataFrame(pipeline_logs)

# Display results
print("PHASE 4 VERIFICATION RESULTS")
display(verification_df)

print("PHASE 4 PIPELINE LOGS")
display(pipeline_df)

# Save results
output_dir = Path("results/phase4")
output_dir.mkdir(parents=True, exist_ok=True)

verification_df.to_csv(
    output_dir / "phase4_verification_results.csv",
    index=False
)

pipeline_df.to_csv(
    output_dir / "phase4_pipeline_logs.csv",
    index=False
)

print("All Phase 4 verification checks passed.")
print("Phase 4 results saved successfully.")


PHASE 4 VERIFICATION RESULTS


,Missingness,Alert,Severity,Values Imputed,Remaining Missing,Verification
0,0.00,False,None,0,0,PASS
1,0.05,False,Below alert threshold,225,0,PASS
2,0.15,True,Moderate,675,0,PASS
3,0.30,True,Severe,1350,0,PASS


PHASE 4 PIPELINE LOGS


,Injected Missingness,feature,observed_missingness,missingness_increase,failure_detected,failure_type,severity,recovery_action,missing_before,missing_after,values_imputed,recovery_applied
0,0.00,BILL_AMT1,0.00,0.00,False,No missingness,None,No correction,0,0,0,False
1,0.05,BILL_AMT1,0.05,0.05,False,Numerical missingness,Below alert threshold,Routine median preprocessing,225,0,225,True
2,0.15,BILL_AMT1,0.15,0.15,True,Numerical missingness,Moderate,Median imputation,675,0,675,True
3,0.30,BILL_AMT1,0.30,0.30,True,Numerical missingness,Severe,Median imputation,1350,0,1350,True


All Phase 4 verification checks passed.
Phase 4 results saved successfully.
